In [1]:
#!/usr/bin/env python3

from pathlib import Path
import xarray as xr

ERA5_ROOT = Path("../data/era5")
DERIVED_ROOT = Path("../data/era5_derived")
ANOM_ROOT = Path("../data/era5_anomalies")
CLIM_ROOT = Path("../data/era5_climatology")

ANOM_ROOT.mkdir(parents=True, exist_ok=True)
CLIM_ROOT.mkdir(parents=True, exist_ok=True)

VARIABLE_DIRS = {
    "geopotential": ERA5_ROOT / "geopotential",
    "vertical_velocity": ERA5_ROOT / "vertical_velocity",
    "specific_humidity": ERA5_ROOT / "specific_humidity",
    "u_component_of_wind": ERA5_ROOT / "u_component_of_wind",
    "v_component_of_wind": ERA5_ROOT / "v_component_of_wind",
    "wind_speed_925": DERIVED_ROOT / "wind_speed_925",
    "moisture_flux_u_925": DERIVED_ROOT / "moisture_flux_u_925",
    "moisture_flux_v_925": DERIVED_ROOT / "moisture_flux_v_925",
    "moisture_flux_mag_925": DERIVED_ROOT / "moisture_flux_mag_925",
}


def open_all_monthly_files(var_dir: Path) -> xr.DataArray:
    files = sorted(var_dir.glob("*.nc"))
    if not files:
        raise FileNotFoundError(f"No files found in {var_dir}")

    ds = xr.open_mfdataset(files, combine="by_coords")
    if len(ds.data_vars) != 1:
        raise ValueError(f"Expected 1 variable in {var_dir}, got {list(ds.data_vars)}")

    var_name = list(ds.data_vars)[0]
    da = ds[var_name].sortby("time")
    return da


for var_name, var_dir in VARIABLE_DIRS.items():
    print(f"Working on {var_name}")

    da = open_all_monthly_files(var_dir)

    # monthly climatology
    clim = da.groupby("time.month").mean("time").rename(var_name)

    # anomalies
    anom = (da.groupby("time.month") - clim).rename(f"{var_name}_anom")

    out_clim_dir = CLIM_ROOT / var_name
    out_anom_dir = ANOM_ROOT / var_name
    out_clim_dir.mkdir(parents=True, exist_ok=True)
    out_anom_dir.mkdir(parents=True, exist_ok=True)

    clim.to_netcdf(
        out_clim_dir / f"{var_name}_monthly_climatology_1980_2024.nc",
        encoding={clim.name: {"zlib": True, "complevel": 4}}
    )
    anom.to_netcdf(
        out_anom_dir / f"{var_name}_monthly_anomalies_1980_2024.nc",
        encoding={anom.name: {"zlib": True, "complevel": 4}}
    )

    print(f"Saved climatology and anomalies for {var_name}")

Working on geopotential


ERROR 1: PROJ: proj_create_from_database: Open of /home/k16v981/.conda/envs/my_env/share/proj failed
HDF5-DIAG: Error detected in HDF5 (1.12.2) thread 0:
  #000: H5A.c line 528 in H5Aopen_by_name(): can't open attribute
    major: Attribute
    minor: Can't open object
  #001: H5VLcallback.c line 1091 in H5VL_attr_open(): attribute open failed
    major: Virtual Object Layer
    minor: Can't open object
  #002: H5VLcallback.c line 1058 in H5VL__attr_open(): attribute open failed
    major: Virtual Object Layer
    minor: Can't open object
  #003: H5VLnative_attr.c line 130 in H5VL__native_attr_open(): can't open attribute
    major: Attribute
    minor: Can't open object
  #004: H5Aint.c line 545 in H5A__open_by_name(): unable to load attribute info from object header
    major: Attribute
    minor: Unable to initialize object
  #005: H5Oattribute.c line 494 in H5O__attr_open_by_name(): can't locate attribute: '_QuantizeBitGroomNumberOfSignificantDigits'
    major: Attribute
    minor: